# Part 2: PDF 추출 방식 비교 + Base RAG

이 섹션에서는:
1. **이론**: 왜 "추출"이 RAG 성능을 좌우하는가
2. **추출 파이프라인 3종**: PyMuPDFLoader, PyMuPDF4LLMLoader, VLM
3. **Base RAG 세팅**: chunk/embedding/LLM 고정
4. **FAISS 기반 답변 비교**: 3종 추출 결과 각각으로 정확도/근거성/속도·비용 비교

## 2-1. 이론: 왜 "추출"이 RAG 성능을 좌우하는가

PDF는 원래 시각적 레이아웃을 위한 포맷입니다. 텍스트만 뽑으면 문제가 생깁니다.

- **레이아웃**: 2단 구성, 페이지 나눔 등으로 문장 순서가 뒤바뀔 수 있음
- **표**: 셀 경계가 무시되면 "25,126 | 27,935"가 "25 126 27 935"처럼 깨짐
- **헤더/푸터**: 매 페이지 반복되는 헤더가 노이즈가 됨
- **줄바꿈/하이픈**: "매출액\n93.8조"가 한 단위로 인식되지 않을 수 있음

따라서 추출 방식을 바꾸면 검색 품질과 최종 답변 정확도가 달라집니다.

In [3]:
# 패키지 임포트
import os
import time
import pickle
from typing import Optional, List

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

True

## 2-2. 추출 파이프라인 3종 구현

In [4]:
# (1) PyMuPDFLoader — 텍스트 중심 추출
pdf_path = "IT_IBKS20260130IBK투자증권.pdf"

loader_pymupdf = PyMuPDFLoader(pdf_path)
docs_pymupdf = loader_pymupdf.load()
print(f"PyMuPDFLoader: {len(docs_pymupdf)}개 페이지")

PyMuPDFLoader: 26개 페이지


In [5]:
# (2) PyMuPDF4LLMLoader — 레이아웃·LLM 친화 Markdown
loader_4llm = PyMuPDF4LLMLoader(pdf_path)
docs_4llm = loader_4llm.load()
print(f"PyMuPDF4LLMLoader: {len(docs_4llm)}개 페이지")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
PyMuPDF4LLMLoader: 26개 페이지


In [6]:
# (3) VLM 기반 추출 — 이미지 → GPT-4o → Markdown
import fitz
import base64
from langchain_core.messages import HumanMessage

def pdf_to_images(pdf_path: str, max_pages: Optional[int] = None, start_page: int = 0) -> List[tuple]:
    doc = fitz.open(pdf_path)
    images = []
    end = min(start_page + (max_pages or len(doc)), len(doc))
    for i in range(start_page, end):
        page = doc[i]
        pix = page.get_pixmap(dpi=200)
        images.append((i + 1, pix.tobytes("png")))
    doc.close()
    return images

def vlm_extract_page(image_bytes: bytes, llm, page_num: int) -> str:
    b64 = base64.standard_b64encode(image_bytes).decode("utf-8")
    prompt = (
        "아래 이미지는 삼성전자 실적 발표 자료 PDF의 한 페이지입니다. "
        "이미지에 보이는 모든 텍스트, 표, 차트, 그래프의 내용을 Markdown 형식으로 작성해주세요. "
        "표는 | 구분자를 사용한 Markdown 테이블로 작성합니다. 수치와 단위를 정확히 기재하세요."
    )
    msg = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}", "detail": "high"}},
    ])
    resp = llm.invoke([msg])
    return resp.content if hasattr(resp, "content") else str(resp)

In [7]:
def load_pdf_with_vlm(pdf_path: str, max_pages: Optional[int] = None, start_page: int = 0, model: str = "gpt-4o"):
    llm = ChatOpenAI(model=model, temperature=0)
    images = pdf_to_images(pdf_path, max_pages, start_page)
    docs = []
    for page_num, img_bytes in images:
        text = vlm_extract_page(img_bytes, llm, page_num)
        docs.append(Document(page_content=text, metadata={"source": pdf_path, "page": page_num}))
    return docs

# VLM 로드 (API 키 필요, 비용 발생)
docs_vlm = load_pdf_with_vlm(pdf_path, max_pages=None)
print(f"VLM: {len(docs_vlm)}개 페이지")

VLM: 26개 페이지


### 추출 결과 비교: 동일 페이지에서 3종 로더의 차이

표가 포함된 페이지(9페이지)를 기준으로, 각 로더가 어떻게 다르게 추출하는지 비교합니다.
- **PyMuPDFLoader**: 텍스트만 추출 → 표 구조 손실 가능
- **PyMuPDF4LLMLoader**: Markdown 형태로 표 구조 보존 시도
- **VLM (GPT-4o)**: 이미지 인식 → 표/차트까지 Markdown으로 변환

In [8]:
# 9페이지(0-based index 8) 비교 — 표가 포함된 페이지
page_idx = 8
char_limit = 800

print("=" * 80)
print("PyMuPDFLoader - 페이지 {} 추출 결과 (앞 {}자)".format(page_idx + 1, char_limit))
print("=" * 80)
print(docs_pymupdf[page_idx].page_content[:char_limit])

print("\n" + "=" * 80)
print("PyMuPDF4LLMLoader - 페이지 {} 추출 결과 (앞 {}자)".format(page_idx + 1, char_limit))
print("=" * 80)
print(docs_4llm[page_idx].page_content[:char_limit])

has_table_md = "|" in docs_4llm[page_idx].page_content
print("\nMarkdown 테이블 구분자 포함: {}".format("예" if has_table_md else "아니오"))

vlm_page_idx = min(page_idx, len(docs_vlm) - 1)
vlm_page_num = docs_vlm[vlm_page_idx].metadata.get("page", vlm_page_idx + 1)
print("\n" + "=" * 80)
print("[VLM] GPT-4o - 페이지 {} 추출 결과 (앞 {}자)".format(vlm_page_num, char_limit))
print("=" * 80)
print(docs_vlm[vlm_page_idx].page_content[:char_limit])
has_vlm_table = "|" in docs_vlm[vlm_page_idx].page_content
print("\n[VLM] Markdown 테이블 구분자 포함: {}".format("예" if has_vlm_table else "아니오"))

PyMuPDFLoader - 페이지 9 추출 결과 (앞 800자)
김운호  6915-5656  
 
 
IBKS RESEARCH │9 
표 1. 삼성전자 분기별 실적 추이 및 전망 
 
(단위: 십억원) 
2025 
2026 
4분기 증감률 
1Q 
2Q 
3Q 
4Q 
1QE 
2QE 
3QE 
4QE 
QoQ(%) 
YoY(%) 
매출액 
DS 
25,126 
27,935 
33,098 
43,983 
56,633 
67,768 
73,008 
74,604 
32.9 
46 
 
Display 
5,931 
6,419 
8,119 
9,528 
6,230 
6,509 
7,991 
7,611 
17.4 
17.5 
 
MX/네트워크 
37,058 
29,208 
34,147 
29,329 
30,457 
25,443 
28,674 
25,346 
-14.1 
13.8 
 
VD/가전 
14,509 
13,882 
13,952 
14,804 
13,150 
13,613 
13,279 
13,504 
6.1 
2.8 
 
HAR 
3,409 
3,818 
4,009 
4,610 
4,841 
5,083 
5,337 
5,604 
15 
17.7 
 
합계 
79,141 
74,566 
86,062 
93,865 
103,089 
110,358 
120,392 
118,931.00 
9.1 
23.9 
영업이익 
DS 
1,128 
405 
6,988 
16,383 
32,154 
42,466 
47,710 
48,694 
134.4 
462.7 
 
Display 
525 
506 
1,207 
2,012 
514 
612 
1,253 
1,155 
66.7 
123.2 


PyMuPDF4LLMLoader - 페이지 9 추출 결과 (앞 800자)
표 1. 삼성전자 분기별 실적 추이 및 전망



김운호 6915-5656



|(단위:십억원)|2025|2026|4분기증감률|
|---|---|---|---|
|(단위: 십억원)|1Q<br>2Q<br>3Q<br>

In [9]:
# 3종 추출본에 동일 TextSplitter 적용
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=180,
    separators=["\n\n", "\n", ". ", ",", " "],
    length_function=len,
)

splits_pymupdf = text_splitter.split_documents(docs_pymupdf)
splits_4llm = text_splitter.split_documents(docs_4llm)
splits_vlm = text_splitter.split_documents(docs_vlm) if docs_vlm else []

print(f"PyMuPDF splits: {len(splits_pymupdf)}")
print(f"4LLM splits: {len(splits_4llm)}")
print(f"VLM splits: {len(splits_vlm)}")

PyMuPDF splits: 85
4LLM splits: 96
VLM splits: 105


## 2-3. Base RAG 세팅 (고정)

- chunk_size=700, chunk_overlap=180
- embeddings: BAAI/bge-m3
- top_k=20
- LLM: ChatOpenAI (gpt-4o 또는 동일 모델)

In [10]:
# 임베딩 모델
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
TOP_K = 20

/var/folders/48/477vrv_n08b185fbd1ycjrqh0000gn/T/ipykernel_19257/1400508315.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [11]:
# LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)
RAG_PROMPT = """다음 맥락을 바탕으로 질문에 답하세요. 맥락에 없는 내용은 추측하지 마세요.

맥락:
{context}

질문: {question}
답변:"""

In [12]:
# Part 1 정답셋 로드 (없으면 인라인 정의)
try:
    with open("test_questions_v2.pkl", "rb") as f:
        test_questions = pickle.load(f)
except FileNotFoundError:
    test_questions = [
        {"question": "삼성전자 2025년 4분기 매출액은?", "expected_answer": "93.8조원", "category": "재무"},
        {"question": "삼성전자 2025년 4분기 영업이익은?", "expected_answer": "20.1조원", "category": "재무"},
        {"question": "삼성전자 DS 부문 2025년 4분기 영업이익은?", "expected_answer": "16.4조원", "category": "재무"},
        {"question": "삼성전자 메모리 사업부 2025년 4분기 영업이익은?", "expected_answer": "17.2조원", "category": "재무"},
        {"question": "SK하이닉스 2025년 4분기 영업이익은?", "expected_answer": "19.2조원", "category": "재무"},
        {"question": "2026년 1분기 삼성전자 영업이익은?", "expected_answer": "35.3조원", "category": "재무"},
        {"question": "2026년 1분기 SK하이닉스 영업이익은?", "expected_answer": "31조원", "category": "재무"},
        {"question": "삼성전자 목표주가는?", "expected_answer": "240,000원", "category": "투자의견"},
        {"question": "SK하이닉스 목표주가는?", "expected_answer": "1,100,000원", "category": "투자의견"},
        {"question": "25년 4분기 영업이익은 삼성전자와 SK하이닉스 중 누가 더 큰가?", "expected_answer": "SK하이닉스", "category": "비교"},
        {"question": "2026년 1분기 삼성전자 메모리 부문 영업이익은?", "expected_answer": "33.1조원", "category": "재무"},
        {"question": "2025년 4분기 DRAM ASP 상승률은 삼성과 SK 중 어느 쪽이 더 높은가?", "expected_answer": "삼성전자", "category": "비교"},
    ]
print(f"질문 수: {len(test_questions)}")

질문 수: 12


In [23]:
# Part 1 평가 함수 (독립 실행용)
import re

def check_accuracy(llm_answer: str, expected_answer: str) -> str:
    ans, exp = llm_answer.strip(), expected_answer.strip()
    if not ans: return 'fail'
    if exp in ans: return 'exact'
    nums_exp = re.findall(r'[\d.,]+', exp)
    nums_ans = re.findall(r'[\d.,]+', ans)
    for n in nums_exp:
        if any(n in a for a in nums_ans): return 'partial'
    return 'fail'

def check_grounded(retrieved_docs, expected_answer: str) -> tuple:
    exp = expected_answer.strip()
    for doc in retrieved_docs:
        idx = doc.page_content.find(exp)
        if idx != -1:
            start = max(0, idx - 100)
            end = min(len(doc.page_content), idx + len(exp) + 100)
            snippet = doc.page_content[start:end]
            return True, f"...{snippet}..."
    return False, ""

def evaluate_rag(question, expected_answer, retrieved_docs, llm_answer, elapsed_sec):
    grounded, evidence = check_grounded(retrieved_docs, expected_answer)
    return {
        "accuracy": check_accuracy(llm_answer, expected_answer),
        "grounded": grounded,
        "grounded_evidence": evidence,
        "elapsed": elapsed_sec,
    }

## 2-4. FAISS VectorStoreRetriever 기반 답변 비교

In [14]:
def run_rag_on_splits(splits, name: str, num_questions: int = 3):
    """splits로 FAISS 구축 후, 처음 num_questions개 질문에 대해 RAG 실행."""
    if not splits:
        print(f"{name}: splits 없음. 건너뜀.")
        return []
    vs = FAISS.from_documents(splits, embeddings)
    retriever = vs.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})
    results = []
    for t in test_questions[:num_questions]:
        q, exp = t["question"], t["expected_answer"]
        docs = retriever.invoke(q)
        context = "\n---\n".join(d.page_content for d in docs)
        t0 = time.time()
        msg = llm.invoke([HumanMessage(content=RAG_PROMPT.format(context=context, question=q))])
        elapsed = time.time() - t0
        ans = msg.content if hasattr(msg, "content") else str(msg)
        ev = evaluate_rag(q, exp, docs, ans, elapsed)
        results.append({"question": q, "expected_answer": exp, "answer": ans[:200], **ev})
    return results

In [24]:
# PyMuPDFLoader 추출본으로 RAG 실행
res_pymupdf = run_rag_on_splits(splits_pymupdf, "PyMuPDFLoader", num_questions=12)
for r in res_pymupdf:
    print(f"Q: {r['question']}")
    print(f"  정답: {r['expected_answer']}")
    print(f"  LLM 답변: {r['answer']}")
    print(f"  정확도: {r['accuracy']}, 근거성: {r['grounded']}, 속도: {r['elapsed']:.2f}s")
    if r['grounded'] and r.get('grounded_evidence'):
        print(f"  근거: \"{r['grounded_evidence']}\"")
    print()

Q: 삼성전자 2025년 4분기 매출액은?
  정답: 93.8조원
  LLM 답변: 삼성전자의 2025년 4분기 매출액은 93.8조원입니다.
  정확도: exact, 근거성: True, 속도: 1.70s
  근거: "...김운호  6915-5656  
 
 
IBKS RESEARCH │7 
 2025년 4분기 매출액은 93.8조원 
삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. 
MX/네트워크를 제외한 전 사업부 매출액은 3분기 대비 증가하였다. DS는 가격..."

Q: 삼성전자 2025년 4분기 영업이익은?
  정답: 20.1조원
  LLM 답변: 삼성전자의 2025년 4분기 영업이익은 20.1조원입니다.
  정확도: exact, 근거성: True, 속도: 4.90s
  근거: "... 복합적 영향으로 부진했고, 디스
플레이는 중소형, 대형 모두 물량이 개선된 영향으로 증가했다. 삼성전자의 
2025년 4분기 영업이익은 2025년 3분기 대비 65.4% 증가한 20.1조원이다. 
DS, 디스플레이를 제외한 나머지 사업부 영업이익은 3분기 대비 감소하였
다. DS는 큰 폭의 가격 상승으로 2배 이상 증가했고, 디스플레이는 소형, 
대형 물량 증가로 ..."

Q: 삼성전자 DS 부문 2025년 4분기 영업이익은?
  정답: 16.4조원
  LLM 답변: 삼성전자 DS 부문의 2025년 4분기 영업이익은 16.4조원입니다.
  정확도: exact, 근거성: True, 속도: 1.01s
  근거: "... 증가, MX는 원가 부담 및 ASP 하락으로 47.6% 감소, VD/가전은 적자폭이 확
대되었다. 사업부별로는  
1) DS 사업부는 2025년 3분기 대비 134.4% 증가한 16.4조원이다. 메모리는 17.2
조원, 비메모리는 0.8조원 수준의 영업적자이다. DRAM, NAND 모두 ASP 상
승 영향을 크게 받을 것으로 분석된다. 비메모리는 물량 부진 및 충당..."

Q: 삼성전자 메모리 사업부 2025년

In [27]:
# PyMuPDF4LLMLoader 추출본으로 RAG 실행
res_4llm = run_rag_on_splits(splits_4llm, "PyMuPDF4LLMLoader", num_questions=12)
for r in res_4llm:
    print(f"Q: {r['question']}")
    print(f"  정답: {r['expected_answer']}")
    print(f"  LLM 답변: {r['answer']}")
    print(f"  정확도: {r['accuracy']}, 근거성: {r['grounded']}, 속도: {r['elapsed']:.2f}s")
    if r['grounded'] and r.get('grounded_evidence'):
        print(f"  근거: \"{r['grounded_evidence']}\"")
    print()

Q: 삼성전자 2025년 4분기 매출액은?
  정답: 93.8조원
  LLM 답변: 삼성전자의 2025년 4분기 매출액은 93.8조원입니다.
  정확도: exact, 근거성: True, 속도: 2.49s
  근거: "...MX 제외 전 사업부

증가. DS가 주도



김운호 6915-5656


2025년 4분기 매출액은 93.8조원


삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다.


MX/네트워크를 제외한 전 사업부 매출액은 3분기 대비 증가하였다. DS는 ..."

Q: 삼성전자 2025년 4분기 영업이익은?
  정답: 20.1조원
  LLM 답변: 삼성전자의 2025년 4분기 영업이익은 20.1조원입니다.
  정확도: exact, 근거성: True, 속도: 1.41s
  근거: "...삼성전자/SK하이닉스: 흔들리지 않는 편안함


2025년 4분기 영업이익은 20.1조원



DS가 주도


8 │ IBKS RESEARCH



삼성전자의 2025년 4분기 영업이익은 2025년 3분기 대비 65.4% 증가한 20.1조원이


다. DS, 디스플레..."

Q: 삼성전자 DS 부문 2025년 4분기 영업이익은?
  정답: 16.4조원
  LLM 답변: 삼성전자 DS 부문의 2025년 4분기 영업이익은 16.4조원입니다.
  정확도: exact, 근거성: True, 속도: 0.86s
  근거: "...가, MX는 원가 부담 및 ASP 하락으로 47.6% 감소, VD/가전은 적자폭이 확


대되었다. 사업부별로는


1) DS 사업부는 2025년 3분기 대비 134.4% 증가한 16.4조원이다. 메모리는 17.2


조원, 비메모리는 0.8조원 수준의 영업적자이다. DRAM, NAND 모두 ASP 상


승 영향을 크게 받을 것으로 분석된다. 비메모리는 물량 부진 ..."

Q: 삼성전자 메모리 사업부 2025년 4분기 영업이익은?
  정답: 17.2조원
  LLM 답변: 삼성전자 메모리 사업부의 2025년

In [28]:
# VLM 추출본으로 RAG 실행 (VLM 로드된 경우만)
res_vlm = run_rag_on_splits(splits_vlm, "VLM", num_questions=12)
for r in res_vlm:
    print(f"Q: {r['question']}")
    print(f"  정답: {r['expected_answer']}")
    print(f"  LLM 답변: {r['answer']}")
    print(f"  정확도: {r['accuracy']}, 근거성: {r['grounded']}, 속도: {r['elapsed']:.2f}s")
    if r['grounded'] and r.get('grounded_evidence'):
        print(f"  근거: \"{r['grounded_evidence']}\"")
    print()

Q: 삼성전자 2025년 4분기 매출액은?
  정답: 93.8조원
  LLM 답변: 삼성전자의 2025년 4분기 매출액은 93.8조원입니다.
  정확도: exact, 근거성: True, 속도: 1.11s
  근거: "...성전자 (005930)

## 메모리가 끌고 가는 실적

### 25년 4분기, 압도적 메모리 실적

삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출액은 3분기 대비 증가하였다. DS는 가격 상승으로 증가, VD/가전은 연말 성수기 효과로 개선되었으나 MX 사업부는 물량 감소와 가격..."

Q: 삼성전자 2025년 4분기 영업이익은?
  정답: 20.1조원
  LLM 답변: 삼성전자의 2025년 4분기 영업이익은 20.1조원입니다.
  정확도: exact, 근거성: True, 속도: 0.85s
  근거: "...의 복합적 영향으로 부진했고, 디스플레이는 감소함. 범용 모두 물량이 개선된 영향으로 증가하였다. 삼성전자의 2025년 4분기 영업이익은 2025년 3분기 대비 65.4% 증가한 20.1조원이다. DS, 디스플레이를 제외한 나머지 사업부 영업이익은 3분기 대비 감소하였다. DS는 큰 폭의 가격 상승으로 인해 이익 증가했고, 디스플레이는 소형, 대형 물량 증가로 66...."

Q: 삼성전자 DS 부문 2025년 4분기 영업이익은?
  정답: 16.4조원
  LLM 답변: 삼성전자 DS 부문의 2025년 4분기 영업이익은 2025년 3분기 대비 134.4% 증가한 16.4조원입니다.
  정확도: exact, 근거성: True, 속도: 1.15s
  근거: "...7% 증가, MX는 일부 부품 및 ASP 하락으로 47.6% 감소, VD/가전은 적자폭이 확대되었다. 사업부별로는

1) DS 사업부는 2025년 3분기 대비 134.4% 증가한 16.4조원이다. 메모리는 17.2조원, 비메모리는 0.8조원 수준의 영업적자이다. DRAM, NAND 모두 ASP

In [29]:
# 3종 추출본 splits 저장 (Part 3~5에서 사용)
with open("splits_v2.pkl", "wb") as f:
    pickle.dump({"pymupdf": splits_pymupdf, "4llm": splits_4llm, "vlm": splits_vlm}, f)
print("splits_v2.pkl 저장 완료.")

splits_v2.pkl 저장 완료.
